In [ ]:
%load_ext autoreload
%autoreload 2

# AlphaZero — Chess

This notebook loads a trained checkpoint, plots training curves, and lets you play against the model.

**Training** is done separately via the script — run it in a terminal and let this notebook consume the results:
```bash
uv run python -m train.run_training --preset S          # or M for a bigger model
uv run python -m train.run_training --preset M --device mps
```
Checkpoints land in `checkpoints/<preset>/chess_iter_NNNN.pt`.

---
## 1. Load checkpoint

In [ ]:
# ── Which run to inspect ───────────────────────────────────────────────────────
PRESET         = "XS"   # "XS" / "S" / "M"
CHECKPOINT_DIR = None   # None → checkpoints/<preset>; or set an explicit path

# MCTS simulations when playing (more = stronger, slower):
#   ~50   fast   (~1 s/move)  good for quick games while the model is still weak
#   ~200  medium (~5 s/move)
#   ~600  strong (~15 s/move) with tree reuse
AI_SEARCHES = 200

In [ ]:
import glob
import json
import os

import chess
import chess.svg
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import SVG, clear_output, display

from chess_game.chess_game import ChessGame
from mcts.mcts import MCTS
from model.model import ResNet


def load_checkpoint(preset=PRESET, directory=None):
    """Load the latest checkpoint for a preset.

    Reads the model architecture straight from the saved args so you never
    have to manually match num_res_blocks / num_hidden to the right preset.
    """
    if directory is None:
        directory = f"checkpoints/{preset.lower()}"
    files = sorted(glob.glob(os.path.join(directory, "chess_iter_*.pt")))
    if not files:
        raise FileNotFoundError(
            f"No checkpoints found in {directory!r}.\n"
            f"Run: uv run python -m train.run_training --preset {preset}"
        )
    path = files[-1]
    ckpt = torch.load(path, weights_only=False, map_location="cpu")
    a = ckpt["args"]

    game  = ChessGame()
    model = ResNet(game, num_res_blocks=a["num_res_blocks"], num_hidden=a["num_hidden"])
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    loss_history = ckpt.get("loss_history", [])

    eval_history = []
    eval_path = os.path.join(directory, "eval_history.json")
    if os.path.exists(eval_path):
        with open(eval_path) as f:
            eval_history = json.load(f)

    return game, model, a, loss_history, eval_history, ckpt["iteration"], path


game, model, args, loss_history, eval_history, iteration, ckpt_path = load_checkpoint(
    preset=PRESET, directory=CHECKPOINT_DIR
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded     : {ckpt_path}")
print(f"Iteration  : {iteration}")
print(f"Model      : {args['num_res_blocks']} res_blocks × {args['num_hidden']} hidden  ({n_params:,} params)")
print(f"Input ch   : {game.num_channels}  |  Action size: {game.action_size}")
if loss_history:
    print(f"Loss       : {loss_history[0]:.4f} → {loss_history[-1]:.4f}  ({len(loss_history)} iters)")
if eval_history:
    last_wr = eval_history[-1]['win_rate']
    print(f"Win rate   : {last_wr:.1%} (latest eval, vs checkpoint {eval_history[-1]['baseline_iter']})")

---
## 2. Training curves

In [ ]:
has_eval = bool(eval_history)
ncols    = 2 if has_eval else 1
fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 4))
if ncols == 1:
    axes = [axes]

# ── Loss ─────────────────────────────────────────────────────────────────────
ax = axes[0]
iters = list(range(1, len(loss_history) + 1))
ax.plot(iters, loss_history, lw=1, alpha=0.35, color="steelblue")

# Smoothed trend line
window = max(1, len(loss_history) // 8)
if len(loss_history) >= window * 2:
    kernel   = np.ones(window) / window
    smoothed = np.convolve(loss_history, kernel, mode="valid")
    ax.plot(
        range(window, len(loss_history) + 1), smoothed,
        lw=2, color="steelblue", label=f"{window}-iter moving avg"
    )
    ax.legend(fontsize=9)

ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
ax.set_title(f"Training loss  ({PRESET} preset, iter {iteration})")
ax.grid(True, alpha=0.3)

# ── Eval win rate ─────────────────────────────────────────────────────────────
if has_eval:
    ax2 = axes[1]
    ev_iters  = [e["iteration"]  for e in eval_history]
    win_rates = [e["win_rate"]   for e in eval_history]
    wins      = [e["wins"]       for e in eval_history]
    draws     = [e["draws"]      for e in eval_history]
    losses    = [e["losses"]     for e in eval_history]
    n_games   = [e["wins"] + e["draws"] + e["losses"] for e in eval_history]

    ax2.bar(ev_iters, [w / n for w, n in zip(wins,   n_games)], label="win",  color="seagreen",  alpha=0.8)
    ax2.bar(ev_iters, [d / n for d, n in zip(draws,  n_games)], label="draw", color="steelblue", alpha=0.8,
            bottom=[w / n for w, n in zip(wins, n_games)])
    ax2.bar(ev_iters, [l / n for l, n in zip(losses, n_games)], label="loss", color="tomato",    alpha=0.8,
            bottom=[(w + d) / n for w, d, n in zip(wins, draws, n_games)])

    ax2.axhline(0.5, color="black", lw=1, ls="--", alpha=0.4, label="50% baseline")
    ax2.set_ylim(0, 1)
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel("Fraction of games")
    ax2.set_title("Eval: new model vs prev checkpoint")
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

# Summary table
if has_eval:
    print(f"{'Iter':>6}  {'vs':>6}  {'W':>4}  {'D':>4}  {'L':>4}  {'Win%':>6}")
    print("-" * 38)
    for e in eval_history:
        print(f"{e['iteration']:>6}  {e['baseline_iter']:>6}  "
              f"{e['wins']:>4}  {e['draws']:>4}  {e['losses']:>4}  "
              f"{e['win_rate']:>6.1%}")

---
## 3. Play against the model

Enter moves in **UCI format**: `e2e4`, `g1f3`, `e1g1` (kingside castle), `e1c1` (queenside castle), `e7e8q` (promotion).

Type `resign` to end the game early.

In [ ]:
# ── Play settings ─────────────────────────────────────────────────────────────
human_plays = chess.WHITE   # chess.WHITE or chess.BLACK

# Override AI search count for this game (defaults to the value set in cell 1)
ai_searches = AI_SEARCHES

In [ ]:
mcts_play = MCTS(game, model=model, num_searches=ai_searches)

board     = chess.Board()
last_move = None
mcts_play._root = None


def show_board(board, last_move=None, caption=""):
    clear_output(wait=True)
    svg = chess.svg.board(
        board,
        lastmove=last_move,
        size=420,
        orientation=human_plays,
    )
    display(SVG(svg))
    if caption:
        print(caption)


side = "White" if human_plays == chess.WHITE else "Black"
show_board(board, caption=f"You play {side}.  AI uses {ai_searches} simulations/move.")

while not board.is_game_over():
    current_player = 1 if board.turn == chess.WHITE else -1

    if board.turn == human_plays:
        legal = {m.uci() for m in board.legal_moves}
        while True:
            raw = input(f"Move {board.fullmove_number} ({side} to move) — UCI or 'resign': ").strip().lower()
            if raw == "resign":
                break
            if raw in legal:
                break
            sample = sorted(legal)[:8]
            print(f"  Not a legal move. Examples: {', '.join(sample)}{'…' if len(legal) > 8 else ''}")

        if raw == "resign":
            show_board(board, last_move, caption="You resigned.")
            break

        move   = chess.Move.from_uci(raw)
        action = game.uci_to_action(raw)
        mcts_play.advance_root(action)
        board.push(move)
        last_move = move
        show_board(board, last_move)

    else:
        ai_side = "Black" if human_plays == chess.WHITE else "White"
        show_board(board, last_move, caption=f"Move {board.fullmove_number} — AI ({ai_side}) thinking…")
        policy = mcts_play.search(board.copy(), current_player)
        action = int(np.argmax(policy))
        uci    = game.action_to_uci(action)
        move   = chess.Move.from_uci(uci)
        mcts_play.advance_root(action)
        board.push(move)
        last_move = move
        show_board(board, last_move, caption=f"AI played: {uci}")

# ── Result ────────────────────────────────────────────────────────────────────
if board.is_game_over():
    outcome = board.outcome()
    result  = board.result()
    if outcome is None:
        msg = f"Draw  ({result})"
    elif outcome.winner == human_plays:
        msg = f"You win!  ({result})"
    elif outcome.winner is None:
        msg = f"Draw  ({result})"
    else:
        msg = f"AI wins  ({result})"
    show_board(board, last_move, caption=msg)
    print(msg)